# Notebook 07 — Phylogenetic Trees: Character-Based Methods

**Module:** 06 — Genetics and Evolution  
**Notebook:** 07 of 12  
**Prerequisites:** NB06 (distance methods), Module 05 NB09 (JC distance)  
**Time estimate:** 90 minutes

> Distance methods discard information by collapsing sequence data into a single number
> per pair. Character-based methods use every column of the alignment directly —
> more information, more power, and (for ML) a statistically consistent estimator.

---
## Step 1 — Motivation

Maximum Parsimony and Maximum Likelihood are the two dominant character-based frameworks.
Most modern phylogenetic papers use ML (RAxML, IQ-TREE) with a substitution model
selected by AIC/BIC. Understanding the logic — how a likelihood is assigned to a tree,
what 'branch support' means, what the Felsenstein zone is — is required to evaluate
any phylogenetics paper critically.

---
## Step 3 — Biological Background

### Maximum Parsimony (MP)

Choose the tree that requires the fewest mutations to explain the observed data.
- **Informative sites:** positions where at least two different bases each appear
  in at least two sequences (sites with unique mutations don't help choose between trees)
- **Fitch algorithm:** efficient bottom-up/top-down pass for computing parsimony score
- **Weakness — the Felsenstein zone:** when two long branches are not sister taxa but
  appear similar due to convergent substitutions, parsimony incorrectly groups them
  (long-branch attraction, LBA). ML is consistent (not fooled) even in the Felsenstein zone.

### Maximum Likelihood (ML)

Choose the tree (topology + branch lengths + model parameters) that maximises the
probability of observing the data given the model.

$$\mathcal{L}(T, \vec{v}, \Theta \mid D) = \prod_{\text{site}} \sum_{\text{internal states}} P(\text{site data} \mid T, \vec{v}, \Theta)$$

**Substitution models (nested, selected by AIC/BIC):**

| Model | Parameters | Notes |
|-------|-----------|-------|
| JC69 | 1 (μ) | Equal rates, equal base freqs |
| K80 (K2P) | 2 | Ti ≠ Tv |
| HKY85 | 5 | Ti ≠ Tv + unequal base freqs |
| GTR | 12 | Most general reversible model |
| GTR+Γ | 13 | + rate variation across sites |
| GTR+Γ+I | 14 | + invariable sites |

### Bayesian Phylogenetics

MrBayes and BEAST2 use MCMC to sample the posterior distribution of trees — natural
uncertainty quantification. Posterior probabilities on branches are analogous to
bootstrap support but interpreted probabilistically.

---
## Step 4 — Mathematical Explanation

**Fitch parsimony algorithm (for one site, multiple sequences):**

Bottom-up pass (post-order traversal):
- For each internal node u with children v₁ and v₂:
  - If S(v₁) ∩ S(v₂) ≠ ∅: S(u) = S(v₁) ∩ S(v₂), no mutation needed
  - Else: S(u) = S(v₁) ∪ S(v₂), +1 mutation counted

Total parsimony score = number of mutations summed over all sites.

**AIC for model selection:** AIC = 2k − 2ln(L̂)
where k = number of parameters, L̂ = maximum likelihood. Lower AIC = better model
relative to complexity.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

In [ ]:
# Cell 6.1 — Parsimony-informative sites
def parsimony_informative_sites(alignment: dict) -> list:
    """
    Identify parsimony-informative sites from a sequence alignment.
    A site is informative if at least two different characters each appear
    in at least two sequences.

    Returns list of column indices that are parsimony-informative.
    """
    seqs = list(alignment.values())
    L = len(seqs[0])
    informative = []

    for j in range(L):
        col = [s[j].upper() for s in seqs]
        from collections import Counter
        counts = Counter(col)
        # At least 2 characters each appearing >= 2 times
        qualifying = [c for c, n in counts.items() if n >= 2]
        if len(qualifying) >= 2:
            informative.append(j)

    return informative

# Test alignment
alignment = {
    'Taxon1': 'ATGCATGCATGC',
    'Taxon2': 'ATGCAAGCATGC',  # position 4: A (was T)
    'Taxon3': 'ATGCATGCAAAC',  # positions 8,9: AA (was AT)
    'Taxon4': 'ATGCAAGCAAAC',  # positions 4,8,9 differ
}
informative = parsimony_informative_sites(alignment)
print(f"Parsimony-informative sites: {informative}")
print(f"Total sites: {len(list(alignment.values())[0])}")
print(f"Informative sites: {len(informative)}")

In [ ]:
# Cell 6.2 — Fitch parsimony algorithm for a fixed topology
def fitch_parsimony_site(site_data: dict, topology: list) -> int:
    """
    Compute parsimony score for a single site on a given topology.

    topology: list of (node_i, node_j, parent) triples defining the tree
              in post-order (leaves first)
    site_data: dict {taxon_name: base}
    Returns: minimum number of changes (parsimony score for this site)
    """
    state_sets = {name: {base} for name, base in site_data.items()}
    score = 0

    for child1, child2, parent in topology:
        s1 = state_sets.get(child1, set())
        s2 = state_sets.get(child2, set())
        intersection = s1 & s2
        if intersection:
            state_sets[parent] = intersection
        else:
            state_sets[parent] = s1 | s2
            score += 1

    return score


def total_parsimony(alignment: dict, topology: list) -> int:
    """Sum Fitch parsimony over all sites."""
    seqs = list(alignment.values())
    names = list(alignment.keys())
    L = len(seqs[0])
    total = 0
    for j in range(L):
        site_data = {names[i]: seqs[i][j] for i in range(len(names))}
        total += fitch_parsimony_site(site_data, topology)
    return total


# Compare two tree topologies for 4 taxa: T1, T2, T3, T4
# Topology A: ((T1,T2),(T3,T4))
topo_A = [('Taxon1', 'Taxon2', 'NodeAB'),
           ('Taxon3', 'Taxon4', 'NodeCD'),
           ('NodeAB', 'NodeCD', 'Root')]

# Topology B: ((T1,T3),(T2,T4))
topo_B = [('Taxon1', 'Taxon3', 'NodeAC'),
           ('Taxon2', 'Taxon4', 'NodeBD'),
           ('NodeAC', 'NodeBD', 'Root')]

score_A = total_parsimony(alignment, topo_A)
score_B = total_parsimony(alignment, topo_B)
print(f"Parsimony score — Topology A ((T1,T2),(T3,T4)): {score_A}")
print(f"Parsimony score — Topology B ((T1,T3),(T2,T4)): {score_B}")
print(f"Most parsimonious: {'A' if score_A <= score_B else 'B'}")

In [ ]:
# Cell 6.3 — AIC model comparison (conceptual)
def aic(log_likelihood: float, n_params: int) -> float:
    return 2 * n_params - 2 * log_likelihood

# Simulated log-likelihoods for different models on the same data
# (in practice these come from tools like IQ-TREE's -m TEST)
models = {
    'JC69':    {'params': 1,  'log_L': -3420.1},
    'K80':     {'params': 2,  'log_L': -3398.5},
    'HKY85':   {'params': 5,  'log_L': -3381.3},
    'GTR':     {'params': 12, 'log_L': -3372.1},
    'GTR+G':   {'params': 13, 'log_L': -3350.8},
    'GTR+G+I': {'params': 14, 'log_L': -3350.2},
}

print(f"{'Model':>12} {'Params':>8} {'LogL':>10} {'AIC':>10} {'ΔAIC':>8}")
print("-" * 52)
aics = {m: aic(v['log_L'], v['params']) for m, v in models.items()}
best_aic = min(aics.values())
for model in models:
    v = models[model]
    a = aics[model]
    print(f"  {model:>10} {v['params']:>8} {v['log_L']:>10.1f} {a:>10.1f} {a-best_aic:>8.1f}")
best = min(aics, key=aics.get)
print(f"\nBest model by AIC: {best}")
print("GTR+G+I barely improves over GTR+G (ΔAIC=1.2) — adding +I not justified.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Felsenstein zone conceptual diagram + AIC bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Felsenstein zone — branch length space
ax = axes[0]
b_range = np.linspace(0, 1.5, 200)

# Probability that parsimony recovers the correct topology (approximate)
# for ((A,B),(C,D)) with A,C having long branches = f(short branch, long branch)
def parsimony_consistency(b_long, b_short=0.05):
    # P(informative site supports true tree) vs P(supports wrong tree)
    # Simplified model: transitions accumulate
    p_long  = 0.75 * (1 - np.exp(-4/3 * b_long))
    p_short = 0.75 * (1 - np.exp(-4/3 * b_short))
    # Parsimony correct when short-branch taxon matches its true sister
    p_correct = (1 - p_long) * (1 - p_short) + p_long * p_short
    p_wrong   = (1 - p_long) * p_short + p_long * (1 - p_short)
    return p_correct / (p_correct + 2 * p_wrong)  # 3 topologies

prob_correct = [parsimony_consistency(b) for b in b_range]
ax.plot(b_range, prob_correct, color='steelblue', lw=2)
ax.axhline(1/3, color='gray', ls='--', lw=1, label='Random (1/3 correct)')
ax.axhline(0.5, color='lightgray', ls=':', lw=1)
ax.fill_between(b_range, prob_correct, 1/3,
                where=[p > 1/3 for p in prob_correct], alpha=0.2,
                color='steelblue', label='Parsimony correct region')
ax.fill_between(b_range, prob_correct, 1/3,
                where=[p <= 1/3 for p in prob_correct], alpha=0.3,
                color='tomato', label='Felsenstein zone (LBA)')
ax.set_xlabel('Long branch length')
ax.set_ylabel('P(parsimony recovers correct tree)')
ax.set_title('Felsenstein zone:\nParsimony inconsistent with very long branches')
ax.legend(fontsize=8)

# Panel 2: AIC scores
ax = axes[1]
model_names = list(aics.keys())
delta_aics = [aics[m] - best_aic for m in model_names]
colors = ['tomato' if d == 0 else 'steelblue' for d in delta_aics]
bars = ax.barh(model_names, delta_aics, color=colors, alpha=0.8)
ax.set_xlabel('ΔAIC (lower = better)')
ax.set_title('Model selection by AIC:\nGTR+Γ is the best-supported model')
for bar, d in zip(bars, delta_aics):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{d:.1f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `fitch_parsimony_site(site_data, topology)`. Trace through the algorithm
   manually for site 4 in the example alignment above.
2. What is long-branch attraction? Given an alignment where two distantly related
   taxa have evolved much faster than others, which method should you use and why?
3. The AIC for GTR+Γ is 6714.6 and for GTR+Γ+I is 6714.4 (ΔAIC = 0.2). Should
   you use the more complex model? What does ΔAIC < 2 typically mean?
4. A phylogenetics paper reports Bayesian posterior probabilities of 0.98 for a
   clade but bootstrap support of only 62%. Is this contradictory? Explain the
   difference between these two support measures.

---
## Quiz — Active Recall

1. What is a parsimony-informative site?
2. Describe the Fitch parsimony algorithm in two sentences.
3. What is the Felsenstein zone? Which phylogenetic method is inconsistent there?
4. What does AIC penalise? Why not always choose the most complex model?
5. Name the two most common ML phylogenetics tools used in modern papers.

---
## Reflection

**Date completed:** ____________________

> *[A paper uses maximum parsimony to infer the phylogeny of a gene family where
> some members have unusually high substitution rates. What concern would you raise
> in peer review?]*

---
*Next: `08_molecular_clocks.ipynb`*